# RAG System for Database Information

This notebook implements a Retrieval-Augmented Generation (RAG) system to answer questions about databases using information from a CSV file.


Let's start by setting up the environment.


In [ ]:
%pip install -qU pandas langchain langchain-community sentence-transformers chromadb langchain-ollama


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
guardrails-ai 0.6.5 requires lxml<5.0.0,>=4.9.3, but you have lxml 5.4.0 which is incompatible.
langchain-openai 0.1.25 requires langchain-core<0.3.0,>=0.2.40, but you have langchain-core 0.3.78 which is incompatible.
mistral-common 1.8.3 requires pillow>=10.3.0, but you have pillow 10.1.0 which is incompatible.
outlines 0.1.11 requires outlines_core==0.1.26, but you have outlines-core 0.2.10 which is incompatible.
pymilvus 2.4.4 requires grpcio<=1.63.0,>=1.49.1, but you have grpcio 1.73.1 which is incompatible.
spacy-transformers 1.3.8 requires transformers<4.50.0,>=3.4.0, but you have transformers 4.55.2 which is incompatible.
tensorboard 2.10.1 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.10.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which 

## 1. Load and Process Data

First, we load the database information from the CSV file into a pandas DataFrame. We will then inspect the data to understand its structure and prepare it for the next steps. Each row in the CSV will be treated as a document.


In [2]:
import pandas as pd

# Load the data
file_path = '/storage03/Saboori/NCIBB/frontend/data/databases_infos.csv'
df = pd.read_csv(file_path)

# Display the first few rows of the dataframe
df.head()


,Name,Short Description,Year,reference,File Size,File Size (KB),Dataset Variables,Data Type,Topics,Description,Score
0,A Comprehensive Dataset of Pattern Electroreti...,336 CSV records with 1354 PERG responses (micr...,2024,https://doi.org/10.13026/d24m-w054,5.7 MB,5700,"PERG signals (microvolts, multiple per eye), A...","signal, table",NaN,The PERG-IOBA Dataset provides a comprehensive...,5
1,AF Termination Challenge Database,ECG recordings created for the Computers in Ca...,2004,https://doi.org/10.13026/C2CC7Z,2.4 MB,2400,"2-channel ECG, QRS annotations, 1-min segments...","signal, table","challenge, atrial fibrillation, ecg","This open-access database contains two-lead, 1...",2
2,AHA Database Sample Excluded Record,Two ECG signals that were excluded from the 19...,2003,https://doi.org/10.13026/C27P4P,9.2 MB,9200,"2-lead ECG, 3-hour recording (last 30 mins ann...","signal, table","american heart association, ecg","This is a two-lead, 3-hour ECG recording exclu...",2
3,A large scale 12-lead electrocardiogram databa...,A 12-lead electrocardiogram database for arrhy...,2022,https://doi.org/10.13026/wgex-er52,5.1 GB,5100000,"12-lead ECG, Raw & denoised, 500 Hz, 10 sec pe...","signal, table","arrhythmia, ecg, machine learning","This database contains 45,152 de-identified, e...",2
4,A multi-camera and multimodal dataset for post...,Multimodal dataset with 166k samples for visio...,2021,https://doi.org/10.13026/fyxw-n385,19.5 GB,19500000,"Depth camera frames, synchronized inertial MoC...","sequence, table","computer vision, inertial motion capture, smar...",A comprehensive multimodal dataset of 14 healt...,1


Now, we'll process the DataFrame to create a list of documents. Each document will represent a row from the CSV file. We will combine the information from all columns into a single text string for each row. This text will be used to generate embeddings.


In [29]:
from langchain.docstore.document import Document

# We will create a document for each row in the dataframe.
# The content of the document will be a concatenation of all columns.
def create_document(row):
    content = ""
    for col, value in row.items():
        content += f"{col}: {value}\\n"
    return content

df['document_content'] = df.apply(create_document, axis=1)

# Create a list of LangChain Document objects
documents = [Document(page_content=content) for content in df['document_content']]

# Let's inspect the first document
print(documents[0].page_content)


Name: A Comprehensive Dataset of Pattern Electroretinograms for Ocular Electrophysiology Research: The PERG-IOBA Dataset\nShort Description: 336 CSV records with 1354 PERG responses (microvolts) from 304 subjects at IOBA. Includes age (years), gender, diagnoses, and visual acuity in logMar scale.\nYear: 2024\nreference: https://doi.org/10.13026/d24m-w054\nFile Size: 5.7 MB\nFile Size (KB): 5700\nDataset Variables: PERG signals (microvolts, multiple per eye), Age, Gender, Diagnoses, Visual acuity (logMAR), Laterality, Visit dates\nData Type: signal, table\nTopics: nan\nDescription: The PERG-IOBA Dataset provides a comprehensive, high-quality pattern electroretinogram database from 304 subjects evaluated at IOBA, Spain, from 2003–2022. Records include raw signals for both eyes, full clinical and demographic metadata, and are tailored to advance ophthalmic research, facilitate new statistical models, and support diagnostics for central retinal/optic nerve disorders. Data are provided in d

## 2. Initialize the Embedding Model

We will use an embedding model from Ollama to convert our text documents into numerical vectors. These vectors capture the semantic meaning of the text. As requested, we will use `embeddinggemma:latest`.


In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize the HuggingFace embedding model
# This will download the model from Hugging Face Hub and cache it locally.
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")


/tmp/ipykernel_1648485/222584880.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
2025-10-08 18:41:59.436959: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-08 18:42:02.042458: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS wh

README.md: 0.00B [00:00, ?B/s]

## 3. Create a Vector Store

With the documents and the embedding model ready, we can now create a vector store. The vector store will store the vector representations of our documents and allow for efficient similarity searches. We will use FAISS (Facebook AI Similarity Search) for this purpose.


In [14]:
from langchain_community.vectorstores import Chroma

# Create the vector store from the documents and embeddings
vectorstore = Chroma.from_documents(documents, embeddings)


## 4. Set up the RAG Chain

Now we will build the RAG chain. This chain will:
1.  Take a user's question.
2.  Use the vector store to retrieve relevant documents.
3.  Use a language model to generate an answer based on the retrieved documents and the original question.



In [48]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

# 1. Create a retriever from the vector store
retriever = vectorstore.as_retriever()

# --- Free-text RAG Chain with Thinking ---

# 2. Define a prompt template for free-text answers that includes a "thinking" step
template = """
First, think step-by-step about how to answer the question based on the provided context. Enclose your thinking process in <thinking>...</thinking> tags.
After your thinking process, provide the final answer to the question.

Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Initialize the LLM with the specified model
llm = Ollama(model="gpt-oss:20b")

# 4. Create the RAG chain for free-text answers
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- JSON RAG Chain for Multiple Datasets ---

# 1. Define the desired JSON data structure for a list of datasets
class Dataset(BaseModel):
    dataset_name: str = Field(description="The name of a suitable dataset.")

class DatasetList(BaseModel):
    datasets: List[Dataset] = Field(description="A list of suitable datasets.")

# 2. Set up a parser for the JSON output
json_parser = JsonOutputParser(pydantic_object=DatasetList)

# 3. Define a prompt template for JSON output, instructing it to find all suitable datasets
json_prompt_template = """
You are a helpful assistant that extracts the names of all suitable datasets from the given context.
Only output the JSON object with the list of dataset names.

Context: {context}
Question: {question}

Format Instructions:
{format_instructions}
"""
json_prompt = ChatPromptTemplate.from_template(
    json_prompt_template,
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

# 4. Create the RAG chain for JSON output
json_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | json_prompt
    | llm
    | json_parser
)


## 5. Running the RAG Pipeline and Getting Structured Output

Now we will run the RAG pipeline and observe its output in three stages:
1.  **Thinking**: The retrieved documents that are passed to the language model as context.
2.  **Free-text Answer**: The original, unstructured answer from the LLM.
3.  **JSON Output**: The structured JSON object containing the dataset name.


In [49]:
import re
import json

# Define the question
question = "which databases are suitable for the analysis of jeartfailure patients with high score"

# --- 1. Retrieve Context ---
retrieved_docs = retriever.invoke(question)

# --- 2. Generate Full Answer (including thinking) ---
full_answer = rag_chain.invoke(question)

# --- 3. Extract Thinking and Final Answer ---
thinking_match = re.search(r'<thinking>(.*?)</thinking>', full_answer, re.DOTALL)
thinking_text = thinking_match.group(1).strip() if thinking_match else "No thinking process found."
final_answer_text = re.sub(r'<thinking>.*?</thinking>', '', full_answer, re.DOTALL).strip()

# --- 4. Generate JSON Output ---
json_answer = json_chain.invoke(question)


# --- Display all results ---

# 1. Retrieved Context
print("--- 1. Retrieved Context ---")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:\\n{doc.page_content}\\n")

# 2. Model's Thinking Process
print("\\n--- 2. Model's Thinking Process ---")
print(thinking_text)

# 3. Final Free-text Answer
print("\\n--- 3. Final Free-text Answer ---")
print(final_answer_text)

# 4. JSON Output
print("\\n--- 4. JSON Output ---")
print(json.dumps(json_answer, indent=2))


--- 1. Retrieved Context ---
Document 1:\nName: Brno University of Technology ECG Quality Database (BUT QDB)\nShort Description: The database is intended for the development and objective comparison of algorithms designed to assess the quality of ECG records. It also enables objective comparison of results between authors.\nYear: 2020\nreference: https://doi.org/10.13026/kah4-0w24\nFile Size: 4.2 GB\nFile Size (KB): 4200000\nDataset Variables: Single-lead ECG, 3-axis accelerometer data, signal-quality class annotations (1,2,3), subject demographics\nData Type: signal, table\nTopics: ECG quality\nDescription: Long-term single-lead ECG and 3-axis accelerometer recordings from 15 subjects in free-living conditions. Designed for evaluating ECG signal quality algorithms, includes detailed expert annotations for varying quality levels and accelerometer data for motion context.\nScore: 5\ndocument_content: Name: Brno University of Technology ECG Quality Database (BUT QDB)\nShort Description: 

In [ ]:
# --- 1b. Model Thinking Tokens: The Full Prompt ---

# Format the documents into a single string
context_string = "\\n\\n".join([doc.page_content for doc in retrieved_docs])

# Create the full prompt
full_prompt = prompt.format(context=context_string, question=question)

print("--- Full Prompt Sent to LLM ---")
print(full_prompt)


--- Full Prompt Sent to LLM ---
Human: 
First, think step-by-step about how to answer the question based on the provided context. Enclose your thinking process in <thinking>...</thinking> tags.
After your thinking process, provide the final answer to the question.

Context: Name: Brno University of Technology ECG Quality Database (BUT QDB)\nShort Description: The database is intended for the development and objective comparison of algorithms designed to assess the quality of ECG records. It also enables objective comparison of results between authors.\nYear: 2020\nreference: https://doi.org/10.13026/kah4-0w24\nFile Size: 4.2 GB\nFile Size (KB): 4200000\nDataset Variables: Single-lead ECG, 3-axis accelerometer data, signal-quality class annotations (1,2,3), subject demographics\nData Type: signal, table\nTopics: ECG quality\nDescription: Long-term single-lead ECG and 3-axis accelerometer recordings from 15 subjects in free-living conditions. Designed for evaluating ECG signal quality al

In [ ]:
# --- 2. Model Thinking Tokens: The Full Prompt ---

# Format the documents into a single string
context_string = "\\n\\n".join([doc.page_content for doc in retrieved_docs])

# Create the full prompt
full_prompt = prompt.format(context=context_string, question=question)

print("--- Full Prompt Sent to LLM ---")
print(full_prompt)


--- Full Prompt Sent to LLM ---
Human: 
First, think step-by-step about how to answer the question based on the provided context. Enclose your thinking process in <thinking>...</thinking> tags.
After your thinking process, provide the final answer to the question.

Context: Name: Brno University of Technology ECG Quality Database (BUT QDB)\nShort Description: The database is intended for the development and objective comparison of algorithms designed to assess the quality of ECG records. It also enables objective comparison of results between authors.\nYear: 2020\nreference: https://doi.org/10.13026/kah4-0w24\nFile Size: 4.2 GB\nFile Size (KB): 4200000\nDataset Variables: Single-lead ECG, 3-axis accelerometer data, signal-quality class annotations (1,2,3), subject demographics\nData Type: signal, table\nTopics: ECG quality\nDescription: Long-term single-lead ECG and 3-axis accelerometer recordings from 15 subjects in free-living conditions. Designed for evaluating ECG signal quality al

In [ ]:
# --- 4. JSON Output ---
json_answer = json_chain.invoke(question)

print("\\n--- JSON Output ---")
import json
print(json.dumps(json_answer, indent=2))


\n--- JSON Output ---
{
  "datasets": [
    {
      "dataset_name": "Brno University of Technology ECG Quality Database (BUT QDB)"
    },
    {
      "dataset_name": "CAST RR Interval Sub-Study Database"
    }
  ]
}
